# Mixture of Experts, Advanced

তিনটি demo, সবগুলোই সরাসরি Phase 03 Lesson 4-এর `MoELayer`/`Expert`-এর উপর ভিত্তি করে তৈরি:

1. Scratch থেকে একটি **Expert-Choice** MoE layer: প্রতিটি **token** তার top-k expert বাছাই করার বদলে, প্রতিটি **expert** batch থেকে তার top-C token বাছাই করে (C = capacity)। মাপা হয় কতগুলো token শেষ পর্যন্ত dropped হয় (কোনো expert বাছাই করেনি) বা multiply-served হয় (একাধিক expert বাছাই করেছে)।
2. সাধারণ top-k token-choice routing-এর বিরুদ্ধে একটি সরাসরি তুলনা, Phase 03 Lesson 4-এ ব্যবহৃত **হুবহু** একই সামান্য initial router bias কৌশলের অধীনে (`router.bias[0]`-এ +1.0)। দেখায় যে Expert-Choice **গঠনগতভাবেই** (by construction) পুরোপুরি সমান expert utilization দেয় — training-এর আগেও সত্য — যেখানে auxiliary loss ছাড়া token-choice সেই collapse প্রবণতা পুনরুৎপাদন করে যা Phase 03 মেপেছিল।
3. Capacity factor পরিবর্তনের সাথে top-k token-choice routing-এর একটি token-dropping-rate simulation, প্রকৃত random router-derived assignment এবং একটি কঠোর per-expert capacity limit ব্যবহার করে, প্রতিটি capacity factor-এ dropped token-এর প্রকৃত মাপা ভগ্নাংশ রিপোর্ট করে।

Runtime: CPU-তে এক মিনিটেরও অনেক কম।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান।

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 0. Shared building block: Phase 03 Lesson 4 থেকে অপরিবর্তিতভাবে পুনর্ব্যবহৃত

`Expert` হলো একটি ছোট FFN (Phase 02-এর FFN sublayer-এর মতোই আকৃতি), এবং `MoELayer` হলো Phase 03 Lesson 4-এর সাধারণ top-k **token-choice** MoE layer — প্রতিটি token নিজের top-k expert বাছাই করে। এটি এখানে অপরিবর্তিতভাবে পুনরুৎপাদন করা হয়েছে যাতে সেকশন 2-এর bias-and-collapse experiment Phase 03-এর পদ্ধতির একটি বিশ্বস্ত পুনরাবৃত্তি হয়। এই cell-এ কোনো demo চলে না।

In [ ]:
# ---------------------------------------------------------------------------
# Shared building block, Phase 03 Lesson 4 থেকে অপরিবর্তিতভাবে পুনর্ব্যবহৃত
# ---------------------------------------------------------------------------

class Expert(nn.Module):
    """একটি expert = একটি ছোট FFN, Phase 02-এর FFN sublayer-এর সাথে আকৃতিতে অভিন্ন।"""

    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class MoELayer(nn.Module):
    """সাধারণ top-k TOKEN-CHOICE MoE layer -- Phase 03 Lesson 4-এর router।

    প্রতিটি token নিজের top-k expert বাছাই করে। এখানে অপরিবর্তিতভাবে
    পুনরুৎপাদন করা হয়েছে যাতে নিচের সেকশন 2-এর bias-and-collapse experiment
    Phase 03-এর পদ্ধতির একটি বিশ্বস্ত পুনরাবৃত্তি হয়, এমন কোনো
    পুনর্বাস্তবায়ন নয় যা ভিন্নভাবে আচরণ করতে পারে।
    """

    def __init__(self, d_model, d_ff, num_experts, top_k=1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(num_experts)])

    def forward(self, x):
        """x: (batch, d_model)। ফেরত দেয় (output, router_probs, chosen_expert_ids)।"""
        router_logits = self.router(x)                       # (batch, num_experts)
        router_probs = F.softmax(router_logits, dim=-1)
        top_k_probs, top_k_indices = router_probs.topk(self.top_k, dim=-1)

        output = torch.zeros_like(x)
        for slot in range(self.top_k):
            expert_ids = top_k_indices[:, slot]                # (batch,) প্রতি token-এ কোন expert
            gate = top_k_probs[:, slot].unsqueeze(-1)           # (batch, 1)
            for expert_id in expert_ids.unique():
                mask = expert_ids == expert_id
                output[mask] += gate[mask] * self.experts[expert_id](x[mask])

        return output, router_probs, top_k_indices

## 1. Scratch থেকে Expert-Choice routing

`ExpertChoiceMoELayer` (Zhou et al., 2022) একই router ও একই `Expert` FFN ব্যবহার করে, কিন্তু বাছাইয়ের দিক উল্টে দেয়: প্রতিটি expert (affinity matrix-এর একটি **column**) affinity score অনুযায়ী নিজের top-C token নেয়, একটি নির্দিষ্ট capacity C পর্যন্ত। ফলে একটি token শূন্য expert (dropped), এক expert, বা একাধিক expert দ্বারা বাছাই হতে পারে। Demo-টি 20টি token ও 4টি expert দিয়ে per-token hit count গণনা করে এবং কতগুলো token dropped / একবার served / multiply-served হলো তা দেখায়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Scratch থেকে Expert-Choice routing
# ---------------------------------------------------------------------------

class ExpertChoiceMoELayer(nn.Module):
    """Expert-Choice MoE (Zhou et al., 2022): EXPERT-রা TOKEN বাছাই করে।

    উপরের MoELayer-এর মতোই একই router + একই Expert FFN, কিন্তু বাছাইয়ের দিক
    উল্টানো: প্রতিটি expert-এর জন্য (affinity matrix-এর একটি COLUMN), affinity
    score অনুযায়ী তার top-C token নাও, একটি নির্দিষ্ট capacity C পর্যন্ত। তাই
    একটি token শূন্য expert (dropped), এক expert, বা একাধিক expert দ্বারা
    বাছাই হতে পারে -- per-token কোনো নিশ্চয়তা নেই, শুধু per-expert।
    """

    def __init__(self, d_model, d_ff, num_experts, tokens_per_expert):
        super().__init__()
        self.num_experts = num_experts
        self.capacity = tokens_per_expert  # C: প্রতিটি expert কতগুলো token নেয়, আগে থেকে নির্দিষ্ট
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(num_experts)])

    def forward(self, x):
        """x: (num_tokens, d_model)। ফেরত দেয় (output, per_token_hit_count)।"""
        num_tokens = x.shape[0]
        affinity = self.router(x)                     # (T, E) raw affinity score
        gate = F.softmax(affinity, dim=-1)             # per-token normalized gate value

        output = torch.zeros_like(x)
        per_token_hit_count = torch.zeros(num_tokens)  # প্রতিটি token-কে কতগুলো expert serve করেছে

        # Capacity প্রকৃতপক্ষে উপলব্ধ token-সংখ্যা ছাড়াতে পারে না।
        C = min(self.capacity, num_tokens)

        for expert_id in range(self.num_experts):
            scores_e = affinity[:, expert_id]                  # এই expert-এর COLUMN: প্রতি token-এ একটি score
            top_c_tokens = scores_e.topk(C).indices             # এই expert-এর বাছাই করা token
            gate_values = gate[top_c_tokens, expert_id].unsqueeze(-1)   # (C, 1)
            output[top_c_tokens] += gate_values * self.experts[expert_id](x[top_c_tokens])
            per_token_hit_count[top_c_tokens] += 1

        return output, per_token_hit_count


def expert_choice_demo():
    print("=" * 70)
    print("1. EXPERT-CHOICE ROUTING, FROM SCRATCH")
    print("=" * 70)

    d_model, d_ff, num_experts = 16, 32, 4
    num_tokens = 20
    top_k_equiv = 1  # top-1 token-choice-এর অধীনে প্রতিটি token গড়ে কতগুলো expert পেত
    capacity = (num_tokens * top_k_equiv) // num_experts  # C = T*k / E

    print(f"num_tokens={num_tokens}, num_experts={num_experts}, capacity C={capacity} "
          f"(= tokens_per_expert)\n")

    ec_moe = ExpertChoiceMoELayer(d_model, d_ff, num_experts, tokens_per_expert=capacity)
    x = torch.randn(num_tokens, d_model)
    output, hit_count = ec_moe(x)

    print(f"input shape:  {tuple(x.shape)}")
    print(f"output shape: {tuple(output.shape)}  (unchanged, same as token-choice would give)\n")

    tokens_dropped = (hit_count == 0).sum().item()
    tokens_multi = (hit_count > 1).sum().item()
    tokens_once = (hit_count == 1).sum().item()
    print("Per-token hit counts (how many experts selected each token):")
    print(f"  {hit_count.long().tolist()}")
    print(f"\n  selected by exactly 1 expert: {tokens_once}/{num_tokens}")
    print(f"  selected by 0 experts (DROPPED, residual-only for this layer): "
          f"{tokens_dropped}/{num_tokens}")
    print(f"  selected by 2+ experts (multiply-served): {tokens_multi}/{num_tokens}")
    print(f"\n-> Every expert processed EXACTLY its capacity of {capacity} tokens (by")
    print("   construction -- top-C always returns exactly C tokens), but because")
    print("   experts choose independently, individual tokens ended up unevenly")
    print("   served: some dropped entirely, some picked up by more than one expert.")


expert_choice_demo()

## 2. একই Phase 03 bias scenario-র অধীনে Expert-Choice বনাম token-choice

দুটি routing scheme-কেই expert 0-এর দিকে একই +1.0 initial router bias দেওয়া হয়। `expert_choice_utilization_at_init` কোনো training ছাড়াই, random initialization-এ Expert-Choice-এর per-expert load গণনা করে — balance এখানে গঠনগত। `train_moe_toy_task` Phase 03 Lesson 4-এর হুবহু training পদ্ধতি পুনরায় চালায় (auxiliary load-balancing loss-সহ ও ছাড়া), শেষ 50 step-এর routing গুনে। `imbalance` পুরোপুরি সমান routing থেকে বিচ্যুতি মাপে (0 = পুরোপুরি balanced), এবং `comparison_demo` তিনটি ফলাফল পাশাপাশি ছাপে।

In [ ]:
# ---------------------------------------------------------------------------
# 2. একই Phase 03 bias scenario-র অধীনে Expert-Choice বনাম token-choice
# ---------------------------------------------------------------------------

def expert_choice_utilization_at_init(num_experts=4, num_tokens=32, d_model=16, d_ff=32):
    """Expert-Choice utilization যাচাই করতে কোনো training-ই লাগে না -- balance
    গঠনগত। একটি apples-to-apples setup-এর জন্য token-choice experiment-এর মতোই
    একই router-bias কৌশল প্রয়োগ করো, তারপর দেখাও যে একটিও gradient step-এর আগে,
    random initialization-এই utilization ইতিমধ্যে পুরোপুরি সমান।"""
    torch.manual_seed(0)
    top_k_equiv = 1
    capacity = (num_tokens * top_k_equiv) // num_experts
    ec_moe = ExpertChoiceMoELayer(d_model, d_ff, num_experts, tokens_per_expert=capacity)

    # Phase 03-এর train_moe_toy_task-এর মতোই একই bias-toward-expert-0 কৌশল,
    # এখানে প্রয়োগ করা হয়েছে যদিও বিষয়টি প্রমাণ করতে Expert-Choice-এর training লাগে না।
    with torch.no_grad():
        ec_moe.router.bias[0] += 1.0

    x = torch.randn(num_tokens, d_model)
    _, hit_count = ec_moe(x)

    # Per-expert top-C selection আবার চালিয়ে হুবহু per-expert load গণনা করো।
    affinity = ec_moe.router(x)
    counts = torch.zeros(num_experts)
    for expert_id in range(num_experts):
        top_c = affinity[:, expert_id].topk(capacity).indices
        counts[expert_id] = len(top_c)
    return counts, capacity


def train_moe_toy_task(use_aux_loss, num_experts=4, steps=600, aux_loss_weight=3.0):
    """Phase 03 Lesson 4-এর example.py থেকে অপরিবর্তিতভাবে পুনরুৎপাদিত, তাই এটি
    সেই হুবহু পদ্ধতির একটি বিশ্বস্ত পুনরাবৃত্তি, কোনো পুনর্বাস্তবায়ন নয়।"""
    torch.manual_seed(0)
    d_model, d_ff = 16, 32
    moe = MoELayer(d_model, d_ff, num_experts, top_k=1)

    # ইচ্ছাকৃতভাবে router-এর initial weight-কে সামান্য expert 0-এর দিকে bias করো,
    # সেই ধরনের ছোট random initialization সুবিধা simulate করতে যা সংশোধন না
    # করলে বাড়তে বাড়তে পূর্ণ collapse-এ পরিণত হয়।
    with torch.no_grad():
        moe.router.bias[0] += 1.0

    true_fn = torch.randn(d_model, d_model)  # একটি নির্দিষ্ট target function যা প্রতিটি expert শিখতে পারত
    optimizer = torch.optim.Adam(moe.parameters(), lr=1e-2)

    expert_counts = torch.zeros(num_experts)
    for step in range(steps):
        x = torch.randn(32, d_model)
        y_true = x @ true_fn

        output, router_probs, chosen = moe(x)
        task_loss = F.mse_loss(output, y_true)

        loss = task_loss
        if use_aux_loss:
            chosen_flat = chosen.squeeze(-1)
            f_i = torch.stack([(chosen_flat == e).float().mean() for e in range(num_experts)])
            P_i = router_probs.mean(dim=0)
            aux_loss = num_experts * (f_i * P_i).sum()
            loss = task_loss + aux_loss_weight * aux_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step >= steps - 50:   # শুধু শেষ 50 step-এর routing গণনা করো
            for e in chosen.squeeze(-1):
                expert_counts[e] += 1

    return expert_counts


def imbalance(counts):
    fractions = counts / counts.sum()
    ideal = 1.0 / len(counts)
    return (fractions - ideal).abs().sum().item()


def comparison_demo():
    print("\n" + "=" * 70)
    print("2. EXPERT-CHOICE vs. TOKEN-CHOICE, SAME ROUTER-BIAS SCENARIO")
    print("=" * 70)
    print("Same +1.0 initial bias toward expert 0 as Phase 03 Lesson 4's collapse")
    print("experiment, applied to both routing schemes.\n")

    num_experts = 4

    # --- Expert-Choice: INITIALIZATION-এ utilization যাচাই, কোনো training লাগে না ---
    ec_counts, capacity = expert_choice_utilization_at_init(num_experts=num_experts)
    print(f"Expert-Choice per-expert token counts, AT INITIALIZATION (capacity C={capacity}):")
    print(f"  " + "".join(f"expert {i}: {int(c):>3}  " for i, c in enumerate(ec_counts)))
    print(f"  deviation from perfectly even routing: {imbalance(ec_counts):.3f}")
    print("  -> Despite the bias toward expert 0, EVERY expert processed exactly")
    print(f"     C={capacity} tokens -- perfect balance, with ZERO training and ZERO")
    print("     auxiliary loss. Balance here is a structural property of top-C")
    print("     selection, not a trained-in behavior.\n")

    # --- Token-choice: Phase 03-এর হুবহু training পদ্ধতি আবার চালাও ---
    print("Token-choice, trained for 600 steps on the same toy regression task,")
    print("counting routing over the FINAL 50 steps (1600 token-routings total):\n")
    counts_no_aux = train_moe_toy_task(use_aux_loss=False, num_experts=num_experts)
    counts_with_aux = train_moe_toy_task(use_aux_loss=True, num_experts=num_experts)

    print(f"{'expert':>13}" + "".join(f"{i:>10}" for i in range(num_experts)))
    print(f"{'no aux loss':>13}" + "".join(f"{int(c):>10}" for c in counts_no_aux))
    print(f"{'with aux loss':>13}" + "".join(f"{int(c):>10}" for c in counts_with_aux))

    no_aux_imb = imbalance(counts_no_aux)
    with_aux_imb = imbalance(counts_with_aux)
    ec_imb = imbalance(ec_counts)
    print(f"\nDeviation from perfectly even routing (0 = perfectly balanced):")
    print(f"  token-choice, no aux loss:   {no_aux_imb:.3f}")
    print(f"  token-choice, with aux loss: {with_aux_imb:.3f}")
    print(f"  Expert-Choice (any bias):    {ec_imb:.3f}  (always exactly 0.000, by construction)")
    print(f"\n-> Token-choice without an auxiliary loss reproduces the collapse Phase 03")
    print(f"   measured (imbalance {no_aux_imb:.3f}, all {int(counts_no_aux.sum().item())} routed")
    print(f"   tokens skewed toward expert 0). Adding the aux loss brings it much closer")
    print(f"   to even ({with_aux_imb:.3f}) but still needs that extra loss term and its")
    print("   weight tuned. Expert-Choice needed neither training nor an aux loss to be")
    print("   perfectly balanced -- it cannot become imbalanced in the first place.")


comparison_demo()

## 3. top-k token-choice routing-এ capacity factor এবং token dropping

`simulate_token_dropping` একটি কঠোর per-expert capacity limit-সহ top-k token-choice routing simulate করে: প্রতিটি token-এর প্রতিটি routing slot স্বাধীনভাবে random-এ একটি expert-এ assign হয়, আগমনের ক্রমে প্রসেস হয়, এবং expert-এর capacity পূর্ণ হয়ে গেলে সেই assignment **dropped** হয় — ঠিক Switch Transformer-style implementation যেভাবে overflow সামলায়। `capacity_factor_demo` 1.0 থেকে 2.0 পর্যন্ত কয়েকটি capacity factor-এ 200টি random trial-এর গড় dropped ভগ্নাংশ ছাপে — বেশি slack মানে কম drop, কিন্তু বেশিরভাগ batch-এ নষ্ট compute।

In [ ]:
# ---------------------------------------------------------------------------
# 3. top-k token-choice routing-এ capacity factor এবং token dropping
# ---------------------------------------------------------------------------

def simulate_token_dropping(num_tokens, num_experts, top_k, capacity_factor, num_trials=200):
    """একটি কঠোর per-expert capacity limit-সহ top-k token-choice routing simulate করে।

    প্রতি token-এর top_k টি routing "slot"-এর প্রতিটি স্বাধীনভাবে random-এ assign
    করা হয় (একটি মোটামুটি ভালোভাবে load-balanced কিন্তু পুরোপুরি সমান নয় এমন
    trained router-এর প্রতিফলন), প্রতি expert-এ একটি নির্দিষ্ট token ক্রমে প্রসেস
    করা হয়, এবং সেই expert-এর capacity-র বাইরের যেকোনো assignment একটি DROP --
    ঠিক Switch Transformer-style implementation যেভাবে overflow সামলায়। Dropped-token
    ভগ্নাংশের একটি স্থিতিশীল অনুমানের জন্য অনেকগুলো random trial-এর গড় নেওয়া হয়।
    """
    ideal_load_per_expert = (num_tokens * top_k) / num_experts
    capacity = max(1, int(capacity_factor * ideal_load_per_expert))

    total_routed = 0
    total_dropped = 0
    for _ in range(num_trials):
        expert_load = torch.zeros(num_experts, dtype=torch.long)
        # Random per-token, per-slot expert assignment (ক্রম = আগমনের ক্রম)।
        assignments = torch.randint(0, num_experts, (num_tokens, top_k))
        for token_idx in range(num_tokens):
            for slot in range(top_k):
                expert_id = assignments[token_idx, slot].item()
                total_routed += 1
                if expert_load[expert_id] < capacity:
                    expert_load[expert_id] += 1  # গৃহীত
                else:
                    total_dropped += 1            # capacity ইতিমধ্যে পূর্ণ -- dropped

    return total_dropped / total_routed, capacity


def capacity_factor_demo():
    print("\n" + "=" * 70)
    print("3. CAPACITY FACTOR AND TOKEN DROPPING IN TOP-K ROUTING")
    print("=" * 70)

    num_tokens, num_experts, top_k = 64, 8, 1
    ideal_load = (num_tokens * top_k) / num_experts
    print(f"num_tokens={num_tokens}, num_experts={num_experts}, top_k={top_k}")
    print(f"ideal (perfectly even) load per expert = {ideal_load:.1f} tokens\n")
    print("capacity = capacity_factor * ideal_load; any token whose assigned expert")
    print("is already full when it arrives is DROPPED (passes through via the")
    print("residual only, no expert computation). Averaged over 200 random trials")
    print("per capacity factor:\n")

    capacity_factors = [1.0, 1.25, 1.5, 2.0]
    print(f"{'capacity_factor':>16}{'capacity/expert':>18}{'dropped_fraction':>18}")
    results = []
    for cf in capacity_factors:
        dropped_fraction, capacity = simulate_token_dropping(
            num_tokens, num_experts, top_k, cf, num_trials=200
        )
        results.append((cf, capacity, dropped_fraction))
        print(f"{cf:>16.2f}{capacity:>18}{dropped_fraction * 100:>17.2f}%")

    first_drop = results[0][2]
    last_drop = results[-1][2]
    print(f"\n-> At capacity_factor={results[0][0]:.2f} (zero slack over the ideal load),")
    print(f"   {first_drop * 100:.2f}% of routed tokens were dropped purely from random")
    print("   imbalance across experts -- there's no room to absorb even ordinary")
    print("   statistical variation in how many tokens land on each expert. As the")
    print(f"   capacity factor increases to {results[-1][0]:.2f}, the dropped fraction falls to")
    print(f"   {last_drop * 100:.2f}%, confirming more slack means fewer drops -- at the direct")
    print("   cost of every expert's buffer being sized for a worst-case load it only")
    print("   occasionally reaches, i.e. wasted compute on most batches.")


capacity_factor_demo()

## সবগুলো demo একসাথে

`main()` উপরের তিনটি demo (Expert-Choice routing, Expert-Choice বনাম token-choice তুলনা, এবং capacity-factor token-dropping simulation) এক টানে একসাথে চালায়। প্রতিটি demo ইতিমধ্যে নিজের cell-এ চলেছে, তাই demo দুবার না চালাতে নিচের call-টি comment করে রাখা হয়েছে।

In [ ]:
def main():
    expert_choice_demo()
    comparison_demo()
    capacity_factor_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন